Reason for Excluding CLASIFFICATION_FINAL

CLASIFFICATION_FINAL was excluded to reduce the possibility of data leakage and to make the model more practically usable. In a real-world prediction setting, the model should rely only on patient-related and clinical input attributes that are directly available at the time of prediction. Excluding this feature makes the ANN more defensible, transparent, and suitable for academic evaluation.

The ANN model is designed as a multilayer feedforward neural network for predicting the binary target variable DIED. After preprocessing, the model receives the selected patient and clinical attributes as input. A dense architecture with three hidden layers is proposed, using 64, 32, and 16 neurons respectively. ReLU activation is used in the hidden layers to capture nonlinear relationships among the features, while a sigmoid activation function is used in the output layer to generate a probability score for death prediction. Dropout layers are added after the first and second hidden layers to reduce overfitting and improve generalization.

The ANN model will be trained as a binary classification model using binary cross-entropy loss and the Adam optimizer. A batch size of 256 and a maximum of 30 epochs are selected to ensure stable and efficient training on the large dataset. Early stopping will be used to prevent overfitting by monitoring validation performance and restoring the best model weights. Since the target variable DIED is imbalanced, class weights will be applied during training so that the minority class receives appropriate importance.

The model will be evaluated using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrix, and loss curves. Among these, recall and F1-score will be given higher importance because the project focuses on identifying mortality risk, and the dataset contains significantly fewer death cases than survival cases.

The ANN implementation begins with loading the cleaned dataset and defining DIED as the binary target variable. The selected 16 input features are then separated by type, and the dataset is split into training, validation, and test sets using stratified sampling. Preprocessing is applied in a leakage-free manner, including scaling of AGE and encoding of MEDICAL_UNIT, while binary features are retained in their existing format. Because the dataset is imbalanced, class weights are applied during training.

A feedforward ANN with three hidden layers is then trained using binary cross-entropy loss and the Adam optimizer. Early stopping is used to prevent overfitting and preserve the best-performing model. Performance is assessed using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix, with greater emphasis on recall and F1-score due to the healthcare risk context. Final evaluation is performed only once on the test set, and the results are documented using metric tables and performance curves.

The preprocessing stage begins by separating the selected 16 input features from the binary target variable DIED. The features are categorized into binary, numerical, and categorical groups. A stratified split is then performed to create training, validation, and test sets in a 70:15:15 ratio while preserving the class distribution of the target variable.

Among the input features, AGE is treated as a numerical variable and standardized, while MEDICAL_UNIT is handled as a categorical variable and one-hot encoded. The remaining binary features are retained in their original 0/1 format. To prevent data leakage, all preprocessing transformations are fitted only on the training set and then applied unchanged to the validation and test sets. Since the target variable is imbalanced, class weights are computed using only the training labels and used during ANN training.

----------------------
----------------------

# ANN Model Building

## Step 1: Data Loading, Feature Selection, and Stratified Splitting

In this step, we load the cleaned COVID-19 dataset, finalize the ANN input features, lock `DIED` as the target variable, and create reproducible train, validation, and test splits.

### Final Modeling Setup

- **Target variable:** `DIED`
- **Excluded feature:** `CLASIFFICATION_FINAL`
- **Input features:** 
  `USMER`, `MEDICAL_UNIT`, `SEX`, `PATIENT_TYPE`, `PNEUMONIA`, `AGE`, 
  `DIABETES`, `COPD`, `ASTHMA`, `INMSUPR`, `HIPERTENSION`, 
  `OTHER_DISEASE`, `CARDIOVASCULAR`, `OBESITY`, 
  `RENAL_CHRONIC`, `TOBACCO`

- **Split strategy:** Stratified `70%` training, `15%` validation, `15%` test

### Importing the necessary Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam


I0000 00:00:1775848282.860331       9 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775848282.954891       9 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775848284.809808       9 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
FEATURE_COLUMNS = [
    'USMER', 'MEDICAL_UNIT', 'SEX', 'PATIENT_TYPE',
    'PNEUMONIA', 'AGE', 'DIABETES', 'COPD', 'ASTHMA',
    'INMSUPR', 'HIPERTENSION', 'OTHER_DISEASE',
    'CARDIOVASCULAR', 'OBESITY', 'RENAL_CHRONIC', 'TOBACCO'
]

TARGET_COLUMN = 'DIED'
RANDOM_STATE = 42

### Load & Validate Data

In [3]:
df = pd.read_csv('Data/Cleaned_CovidData.csv')

required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f'Missing columns: {missing_columns}')

print('Dataset shape:', df.shape)
print('Total missing values:', int(df[required_columns].isnull().sum().sum()))

print('\nTarget distribution:')
print(df[TARGET_COLUMN].value_counts().sort_index())

Dataset shape: (1024829, 18)
Total missing values: 0

Target distribution:
DIED
0    950217
1     74612
Name: count, dtype: int64


### Train/Val/Test Split

In [4]:
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

train_df = X_train.copy()
train_df[TARGET_COLUMN] = y_train.to_numpy()

validation_df = X_val.copy()
validation_df[TARGET_COLUMN] = y_val.to_numpy()

test_df = X_test.copy()
test_df[TARGET_COLUMN] = y_test.to_numpy()

### Preprocessing

In [5]:
binary_features = [
    'USMER',
    'SEX',
    'PATIENT_TYPE',
    'PNEUMONIA',
    'DIABETES',
    'COPD',
    'ASTHMA',
    'INMSUPR',
    'HIPERTENSION',
    'OTHER_DISEASE',
    'CARDIOVASCULAR',
    'OBESITY',
    'RENAL_CHRONIC',
    'TOBACCO'
]

numerical_features = ['AGE']
categorical_features = ['MEDICAL_UNIT']

print("Binary features:", len(binary_features))
print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Binary features: 14
Numerical features: ['AGE']
Categorical features: ['MEDICAL_UNIT']


In [6]:
# Step 2.2: Build the preprocessing pipeline

try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', encoder, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")


Preprocessing completed successfully.


In [7]:
# Step 2.3: Inspect transformed data

feature_names = preprocessor.get_feature_names_out()

print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("Processed feature count:", len(feature_names))

pd.DataFrame({
    'Processed_Feature_Name': feature_names
}).head(30)


Processed training shape: (717380, 28)
Processed validation shape: (153724, 28)
Processed test shape: (153725, 28)
Processed feature count: 28


,Processed_Feature_Name
0,num__AGE
1,cat__MEDICAL_UNIT_1
2,cat__MEDICAL_UNIT_2
3,cat__MEDICAL_UNIT_3
4,cat__MEDICAL_UNIT_4
5,cat__MEDICAL_UNIT_5
6,cat__MEDICAL_UNIT_6
7,cat__MEDICAL_UNIT_7
8,cat__MEDICAL_UNIT_8
9,cat__MEDICAL_UNIT_9


In [8]:
# Step 2.4: Convert data type for TensorFlow compatibility

X_train_processed = X_train_processed.astype('float32')
X_val_processed = X_val_processed.astype('float32')
X_test_processed = X_test_processed.astype('float32')

y_train_array = y_train.to_numpy(dtype='float32')
y_val_array = y_val.to_numpy(dtype='float32')
y_test_array = y_test.to_numpy(dtype='float32')

print(X_train_processed.dtype, y_train_array.dtype)


float32 float32


In [9]:
# Step 2.5: Compute class weights from the training set only

classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = {
    0: float(weights[0]),
    1: float(weights[1])
}

print("Class weights:", class_weight_dict)


Class weights: {0: 0.53926019917252, 1: 6.8677720762809225}


In the preprocessing stage, the selected input features were grouped into binary, numerical, and categorical types. The binary features were retained in their original 0/1 form, the numerical feature AGE was standardized, and the categorical feature MEDICAL_UNIT was transformed using one-hot encoding. To prevent data leakage, the preprocessing pipeline was fitted only on the training data and then applied unchanged to the validation and test sets. Since the target variable DIED was imbalanced, class weights were computed using only the training labels so that the ANN could give appropriate importance to the minority class during training.

-------
-----

## Step 3: building and compiling the ANN model

In [10]:
# Step 3.1: Set random seed and define the ANN architecture

tf.keras.utils.set_random_seed(42)

input_dim = X_train_processed.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation='relu'),
    Dropout(0.30),
    Dense(32, activation='relu'),
    Dropout(0.20),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

print("Input dimension:", input_dim)


Input dimension: 28


E0000 00:00:1775848290.838136       9 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [11]:
# Step 3.2: Compile the ANN model

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)


In [12]:
# Step 3.3: View model summary

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,481 (17.50 KB)

 Trainable params: 4,481 (17.50 KB)

 Non-trainable params: 0 (0.00 B)

In this step, a feedforward Artificial Neural Network was constructed for the binary classification task of predicting the target variable DIED. The model architecture consisted of an input layer followed by three dense hidden layers containing 64, 32, and 16 neurons respectively. ReLU activation was used in the hidden layers to capture nonlinear relationships in the input data, while a sigmoid activation function was used in the output layer to produce a probability value for the binary outcome.

To reduce overfitting, dropout layers with rates of 0.30 and 0.20 were added after the first and second hidden layers. The model was compiled using the Adam optimizer with a learning rate of 0.001 and binary cross-entropy as the loss function. During training, the model was configured to monitor accuracy, precision, recall, and AUC.

------------
------------

### step 4: training the ANN with early stopping, validation monitoring, and class weights.

In [13]:
# Step 4.1: Imports, callbacks, and MLflow setup

import matplotlib.pyplot as plt
from pathlib import Path

from tensorflow.keras.callbacks import EarlyStopping

try:
    import mlflow
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "mlflow is not installed in the active notebook kernel. "
        "Install it in your active environment or run this notebook from your Docker environment."
    )

TRAINING_ARTIFACT_DIR = ROOT_DIR / 'Model' / 'artifacts' / 'step4_training'
TRAINING_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_DIR = ROOT_DIR / 'Model' / 'mlruns'
MLFLOW_TRACKING_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(MLFLOW_TRACKING_DIR.resolve().as_uri())
mlflow.set_experiment("Covid_ANN_DIED_Prediction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())


NameError: name 'ROOT_DIR' is not defined